# PaperText Dataset Experiment

**Dataset:** PaperText (Academic Papers - Text)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Span F1

**Description:** Academic papers with text-focused questions

**Documents:** 7 example PDFs
- 1705.07830
- 1801.05147
- 1809.01202
- 1810.08699
- 1909.00754
- 1912.01214
- 2001.03131

**Parameters:**
- Chunk size: 3000 characters
- Chunk overlap: 300 characters
- Top-K retrieval: 5
- Temperature: 0.1
- Max tokens: 512

## Setup and Imports

In [1]:
import sys
import os

# CRITICAL: Change to project root directory
# The preprocess module uses relative paths from project root
project_root = os.path.abspath('../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess, llm
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ All imports successful


## Configuration

In [2]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [3]:
# Experiment Parameters
DATASET_NAME = "paper_text"
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300
TOP_K = 5
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/1_without_optimization/papertext/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: paper_text
Chunk size: 3000
Top-K: 5
Output dir: ./experiments/papertext/results


## Initialize Models

In [4]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model (local, free)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

✓ Together AI client initialized


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized


## Helper Functions

In [5]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """Retrieve context and generate answer"""
    # Retrieve
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # Build prompt
    llm_message = llm.make_prompt(
        question=question,
        context=context,
        task_name=DATASET_NAME,
        llm_type="gpt-4"
    )
    
    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=llm_message,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

✓ Helper functions defined


## Load Q&A Data

In [6]:
# Load PaperText Q&A
csv_file = "./dataset/qa/paper_text_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# IMPORTANT: Filter to only documents with available PDFs in example directory
# Only these 7 PDFs are in dataset/src_doc_files_example/paper_docs/
AVAILABLE_DOCS = [
    "1705.07830",
    "1801.05147",
    "1809.01202",
    "1810.08699",
    "1909.00754",
    "1912.01214",
    "2001.03131"
]

# Filter Q&A dict to only available documents
qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

# Count Q&A per available document
total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")

Total documents in CSV: 1087
Available PDFs: 7

Filtered to documents with PDFs:

  1705.07830: 1 Q&A pairs
  1801.05147: 1 Q&A pairs
  1809.01202: 1 Q&A pairs
  1810.08699: 3 Q&A pairs
  1909.00754: 2 Q&A pairs
  1912.01214: 3 Q&A pairs
  2001.03131: 2 Q&A pairs

Total Q&A to process: 13


## Main Processing Loop

**This will process all documents and Q&A pairs**

**Expected runtime:** Variable based on Q&A count

In [7]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index
    print("Building vector index...")
    collection = build_index(text_chunks, collection_name=f"papertext_{doc_name}")
    print("✓ Index built")
    
    # Process each question
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")


Processing: 1912.01214
PDF: dataset/src_doc_files_example/paper_docs/1912.01214.pdf
Extracting text...
Created 14 chunks
Building vector index...
✓ Index built

Answering 3 questions...

[1/3] which multilingual approaches do they compare with?...
   Answer: The answer is: Cross-lingual Transfer (Kim, Gao, and Ney 2019), MNMT (Johnson et...

[2/3] what are the pivot-based baselines?...
   Answer: ...

[3/3] which datasets did they experiment with?...
   Answer: The answer is: Europarl and MultiUN...

✓ Completed 1912.01214: 3 questions processed

Processing: 1810.08699
PDF: dataset/src_doc_files_example/paper_docs/1810.08699.pdf
Extracting text...
Created 12 chunks
Building vector index...
✓ Index built

Answering 3 questions...

[1/3] what ner models were evaluated?...
   Answer: The answer is: Stanford NER, spaCy 2.0, and Char-biLSTM+biLSTM+CRF...

[2/3] what is the source of the news sentences?...
   Answer: The answer is: The specific source of the news sentences for the gold-stan

## Diagnostic: Check Empty Responses

In [8]:
if all_results:
    import pandas as pd
    
    # Create DataFrame for analysis
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)
    
    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")
    
    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
        
        print(f"\nSample empty questions (first 5):")
        empty_df = results_df[results_df['is_empty']].head(5)
        for idx, row in empty_df.iterrows():
            print(f"\n  [{idx+1}] {row['question'][:80]}...")
            print(f"      Doc: {row['doc']}")
            print(f"      Response: '{row['response']}'")
    else:
        print("\n✓ All questions received answers!")
    
    print(f"\nSample answered questions (first 3):")
    answered_df = results_df[~results_df['is_empty']].head(3)
    for idx, row in answered_df.iterrows():
        print(f"\n  Q: {row['question'][:80]}...")
        print(f"  A: {row['response'][:100]}...")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 13
Empty responses: 1 (7.7%)
Answered: 12 (92.3%)

Empty responses by document:
  1912.01214: 1/3 empty (33.3%)
  1810.08699: 0/3 empty (0.0%)
  1801.05147: 0/1 empty (0.0%)
  1809.01202: 0/1 empty (0.0%)
  2001.03131: 0/2 empty (0.0%)
  1909.00754: 0/2 empty (0.0%)
  1705.07830: 0/1 empty (0.0%)

Sample empty questions (first 5):

  [2] what are the pivot-based baselines?...
      Doc: 1912.01214
      Response: ''

Sample answered questions (first 3):

  Q: which multilingual approaches do they compare with?...
  A: The answer is: Cross-lingual Transfer (Kim, Gao, and Ney 2019), MNMT (Johnson et al. 2016), MNMT Agr...

  Q: which datasets did they experiment with?...
  A: The answer is: Europarl and MultiUN...

  Q: what ner models were evaluated?...
  A: The answer is: Stanford NER, spaCy 2.0, and Char-biLSTM+biLSTM+CRF...


## Evaluate Results

In [9]:
if all_results:
    print("\nEvaluating PaperText results (Span F1)...")
    eval_main(DATASET_NAME, all_results)
else:
    print("❌ No results to evaluate")


Evaluating PaperText results (Span F1)...
{'Answer F1': 0.43077007533529277, 'Missing predictions': 0}


## Save Results

In [10]:
if all_results:
    # Save to CSV
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"papertext_results_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/papertext/results/papertext_results_20260629_104112.csv
Total Q&A: 13

Results by document:
  1912.01214: 3 questions
  1810.08699: 3 questions
  1801.05147: 1 questions
  1809.01202: 1 questions
  2001.03131: 2 questions
  1909.00754: 2 questions
  1705.07830: 1 questions


## Summary Statistics

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    empty_count = results_df['response'].str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    print(f"\n{'='*80}")
    print(f"STATISTICS")
    print(f"{'='*80}")
    print(f"Total questions: {len(results_df)}")
    print(f"Answered: {answered_count} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"Empty responses: {empty_count} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"Avg response length: {results_df['response'].str.len().mean():.0f} characters")


STATISTICS
Total questions: 13
Answered: 12 (92.3%)
Empty responses: 1 (7.7%)
Avg response length: 107 characters


---

## Done!

**Results saved to:** `./experiments/papertext/results/papertext_results_[timestamp].csv`

**Metric:** Span F1

**Next steps:**
1. Review the accuracy score
2. Compare with other datasets
3. Try parameter optimization (chunk_size, top_k, temperature)